# Extending REPLICA

Bring your own feature model, features or dataset, then compose them into a
study recipe. The in-memory examples run without downloads or optional model
backends. Their code cells are executed by
[`test_tutorial_extending_replica.py`](../../tests/docs/test_tutorial_extending_replica.py).

| Interface | What it provides |
|---|---|
| `Dataset` | Stimuli, annotation layers and aligned responses |
| `FeatureModel` | Raw per-unit outputs from a model or computation |
| `Feature` | Per-unit numeric predictors or controls from those outputs or annotations |
| `FrequencySource` | A unigram probability lookup used by frequency features |
| `Study` | A named recipe composing the dataset, features and analysis |

These interfaces support comprehension models beyond language-model surprisal.
`evaluate()` fits and compares response models using held-out log scores and R²;
its result also records the focal coefficients and response definition.


In [ ]:
import numpy as np

from replica import (
    OLS,
    AnalysisConfig,
    EvaluationSpec,
    ItemSplitter,
    LogFrequency,
    ModelFeature,
    UnigramSurprisal,
    UnitLength,
    evaluate,
)
from replica.core import Datum
from replica.dataset.base import SPRDataset
from replica.dataset.contracts import DatasetInfo
from replica.feature import (
    FeatureBlock,
    FeatureModelOutput,
    FeatureValues,
    Surprisal,
    resolve,
)
from replica.studies import Study

## Part A: a new dataset

A dataset subclasses one of the paradigm bases (`SPRDataset`,
`EyeTrackingDataset`, `EEGDataset`, `TabularDataset`) and implements `load()`,
which fills `self._stimuli` and `self._data`. Use `self._make_stimulus(units,
stimulus_id=...)` to build a stimulus with its `words` layer, and append one
`Datum` per participant-measure.

The synthetic example below defines its own whitespace-separated units. A real loader must align the source measurement identifiers to spans in the actual stimulus text. Preserve source interest areas, including multiword or unspaced units; do not replace them with `text.split()` to match an expected word count.

In [ ]:
class DemoSPR(SPRDataset):
    """Toy self-paced-reading corpus built in memory (no files)."""

    info = DatasetInfo(name="demo_spr", language="en", paradigm="self-paced-reading")

    def load(self, path: str) -> None:
        rng = np.random.default_rng(0)
        words = ["the", "quick", "brown", "fox", "jumps"]
        for i in range(6):
            stim = self._make_stimulus(words, stimulus_id=f"demo_{i}")
            self._data.append(
                Datum(
                    stimulus=stim,
                    measure_name="spr_rt",
                    values=rng.uniform(200, 600, size=len(words)),
                    participant_id="p0",
                )
            )


ds = DemoSPR()
ds.load(path="")
print(len(ds.stimuli()), "stimuli;", len(ds.data("spr_rt")), "measure records")

**Where it lives.** A loader reads the source files under `path` and
keeps its declared `self.info` unless it explicitly selects a metadata file with
`self._load_metadata(path, metadata_path=yaml_path)`. For bundled release metadata,
pass the release's catalog path to `find_dataset_yaml` so copied data directories
retain their metadata and checksum checks. For a custom loader's `load` method:

```python
from pathlib import Path

self._load_metadata(path, metadata_path=Path(path) / "metadata.yaml")
```

To group related stimuli from one source material, declare `material_namespace`
and an ordered `material_fields` list in the YAML or in-memory `DatasetInfo`.
`_make_stimulus` derives the group from those stimulus metadata fields; an
explicit `cv_group` takes precedence. Without a declaration, each stimulus
is its own material.

These edits register a bundled loader:

- `dataset/metadata/yaml/self-paced-reading/demo_spr.yaml` holds the metadata (`url` =
  paper DOI, `download_url` = data source, `citation` = bibtex key, and
  `language:` explicitly for non-English corpora, it defaults to `en`).
- `dataset/catalog/registry.py` gains one `DatasetRegistration` in `_BASE_ENTRIES` so
  `registry.get("demo_spr").factory()` loads it from its canonical path:

  ```python
  DatasetRegistration(
      name="demo_spr",
      loader_module="replica.dataset.spr.demo_spr",
      loader_class="DemoSPR",
      default_subdir="self-paced-reading/original/demo_spr",
      paradigm="self-paced-reading",
  )
  ```
- `dataset/catalog/sources.py` gains a `DATASETS` entry under the same key,
  naming the download source and files (or `"source": "manual"` with access
  instructions), so `replica fetch demo_spr` places the data at that path.
- `dataset/catalog/dataset_counts.json`, the committed counts snapshot the datasets
  page reads, gains the new key: with the data on disk, run
  `replica datasets --facts --measure --write-snapshot --only-missing`. The
  docs build fails while a registered key is missing from the snapshot.

For an eye-tracking event adapter, reuse the event mapping in `dataset/loading/reaggregation.py` and the shared fixation calculator. Preserve event order, trial identity and available spatial precision. Test native-unit round trips against independently released measures, including missingness and both skip settings, before enabling custom units. First-fixation, first-run and go-past summaries must never be summed. See [segmentation and response calculation](segmentation.ipynb) for source declarations and versioned calculation selection.

## Part B: features and feature models

A `Feature` declares `name`, an ordered tuple of `dependencies`, and
`resolve(context, inputs)`. The executor resolves dependencies first. The result
contains indexed blocks; each block distinguishes observations from value axes.
Targets, predictors and controls use this same protocol.

A `FeatureModel` supplies raw output records. It implements `name`, `vocab` and
`__call__(stimulus, extract=None, target_layer="words")`, returning one
`FeatureModelOutput` per unit. An extractor such as `Surprisal()` computes a
quantity from those records. `ModelFeature(model, extractor)` binds them into a
feature. Neither protocol requires inheritance.

The following feature counts runs of vowel letters. It is an orthographic toy
quantity, not a validated syllable counter.


In [ ]:
class VowelGroupCount:
    name = "vowel_group_count"
    dependencies = ()

    def resolve(self, context, inputs):
        blocks = []
        for stimulus in context.stimuli:
            layer = context.target_layer(stimulus)
            counts = []
            for word in stimulus.layers[layer].values:
                groups, previous_vowel = 0, False
                for letter in word.lower():
                    is_vowel = letter in "aeiou"
                    groups += is_vowel and not previous_vowel
                    previous_vowel = is_vowel
                counts.append(float(groups))
            blocks.append(
                FeatureBlock(
                    np.array(counts),
                    context.unit_index(stimulus, layer),
                    stimulus=stimulus,
                    layer=layer,
                )
            )
        return FeatureValues(
            tuple(blocks),
            provenance={
                "outcome_identity": "demo:vowel_group_count:v1",
                "input_scope": "material",
            },
        )


resolve(ds, VowelGroupCount()).blocks[0].values

Built-in `UnitLength` also uses the stimulus directly. Model-backed `Surprisal`
reads `log_p_observed` and negates it. The small unigram model below supplies
that key using a normalized distribution over the five demo words. Its first
word has a probability because this model does not require a preceding context.
Unknown words return `NaN`; a real model must declare its vocabulary policy.


In [ ]:
class TinyUnigramModel:
    name = "demo_unigram"
    vocab = None  # This example supplies observed probabilities only.
    probabilities = {"the": 0.4, "quick": 0.1, "brown": 0.15, "fox": 0.2, "jumps": 0.15}

    def __call__(self, stimulus, extract=None, target_layer="words"):
        return [
            FeatureModelOutput(
                values={
                    "log_p_observed": np.log(
                        self.probabilities.get(word.lower(), np.nan)
                    )
                }
            )
            for word in stimulus.unit_strings(target_layer)
        ]


custom_spec = EvaluationSpec(
    predictors=[ModelFeature(TinyUnigramModel(), Surprisal())],
    controls=[VowelGroupCount(), UnitLength()],
    targets=[ds.feature("spr_rt")],
)
config = AnalysisConfig(
    model=OLS(), design=None, cv=ItemSplitter(3), transform_dv=np.log2
)
custom_results = evaluate(ds, custom_spec, config)
for result in custom_results:
    print(result.feature_name, result.delta_ll_per_unit, result.delta_r2)

This one-reader synthetic dataset explicitly uses OLS and stimulus folds to
exercise the interfaces. It does not establish a scientific effect. The
[getting-started tutorial](getting_started.ipynb) explains participant/item
random effects for individual responses.

Model outputs and feature arrays must match the target layer length. Keep
undefined positions as `NaN` so alignment is preserved; evaluation records row
exclusions. A feature needing `next_log_probs`, such as `Entropy`, raises
`FeatureNotAvailable` when the model does not supply that distribution.

**Where it lives.** Implement a reusable feature in its concept package and
export it from `replica.feature`. Model-derived and stimulus-derived features
use the same public `Feature` protocol; the distinction is which inputs they
consume. Stimulus features resolve directly and do not require a placeholder model. Do not put an unrelated new quantity under frequency or
surprisal merely because those packages already exist.

### A custom frequency source

A `FrequencySource` implements `probability(word)` and `language`.
It returns a probability, with `NaN` for unavailable words. `LogFrequency`
applies log10; `UnigramSurprisal` applies negative log2. The log base belongs
to the feature, not the source.


In [ ]:
MY_TABLE = {"the": 0.05, "cat": 0.001}  # Entries from a larger unigram distribution.


class MyFreq:
    language = "en"

    def probability(self, unit: str) -> float:
        return MY_TABLE.get(unit.lower(), float("nan"))


frequency_stimulus = DemoSPR()._make_stimulus(["The", "cat"], stimulus_id="frequency")
print("LogFrequency (log10):", LogFrequency(MyFreq())(frequency_stimulus, None))
print("UnigramSurprisal (bits):", UnigramSurprisal(MyFreq())(frequency_stimulus, None))

## Part C: the study

A `Study` names the paper, declares its data dependencies, and returns the three
pieces `evaluate()` needs. `paper` is derived from `bibtex`, so the citation is
never hand-typed twice. The one-reader demo declares OLS, log2 responses and
stimulus folds explicitly. Published recipes must specify their method instead
of inheriting the general reanalysis defaults.

In [ ]:
class Demo2099Reading(Study):
    name = "demo2099reading"
    bibtex = r"""
    @article{demo2099reading,
      title = {A Demo Study for the Extending-the-REPLICA Tutorial},
      author = {Doe, Jane},
      year = {2099}
    }
    """
    dependencies = []  # a real study lists DataDependency entries here

    def dataset(self, **kwargs):
        ds = DemoSPR()
        ds.load(path="")
        return ds

    def evaluation_spec(self, *, dataset, **kwargs):
        return EvaluationSpec(
            predictors=[VowelGroupCount()],
            controls=[UnitLength()],
            targets=[dataset.feature("spr_rt")],
        )

    def analysis_config(self):
        return AnalysisConfig(
            model=OLS(), design=None, cv=ItemSplitter(3), transform_dv=np.log2
        )


print(Demo2099Reading().paper)

This study selects stored reading times as its target and stimulus properties
as predictors and controls. Run the composed recipe with one loaded dataset:


In [ ]:
study = Demo2099Reading()
dataset = study.dataset()
spec = study.evaluation_spec(dataset=dataset)
results = evaluate(dataset, spec, study.analysis_config())
for r in results:
    print(f"{r.feature_name} -> {r.dv_name}: delta_ll = {r.delta_ll:.4f}")

**Where it lives.** A study is a package directory: put the class in
`studies/demo2099reading/__init__.py` and write `studies/demo2099reading/README.md`
beside it, stating what the study reproduces and against which anchor.
The README includes the paper title, Abstract, Predictors and BibTeX. A study
with claims also has `EVIDENCE.md` with Headline claims, Scope of reproduction,
Notes and any stage results. Keep section names unique across the two files.
Citation text must match the study's `bibtex`; quoted abstracts need a source
record, or the Abstract section can link to the paper. Register claims in
`studies/demo2099reading/claims.yaml`, then generate the claims section and
navigation with `python scripts/readmes/regen_study_readmes.py --study demo2099reading`.

No registration step follows: `replica.studies` discovers every subpackage
that ships a README, so `ALL_STUDIES`, `STUDY_IDS` and the
`replica.studies.demo2099reading` instance exist as soon as the package and
its README do. The study class is found by name: it must lowercase to the
package name, `Demo2099Reading` for `demo2099reading`. Real dependencies are `DataDependency`
entries (git checkout, OSF, URL, or GitHub release asset) that `ensure_data()`
fetches and checksums; `dependencies = repo_dependencies("<repo>")` reuses a
pinned external-repo checkout.

## Part D: anchor it with a reproduction test

A study is integrated in REPLICA once a test pins its numbers so a future refactor
cannot silently move them. The bar is one of:

- a number the paper itself reports, or
- a number from a direct (non-REPLICA) reimplementation of the paper's recipe on the
  paper's data, stored as a fixture.

The test lives in `tests/studies/demo2099reading/test_reproduction.py` and compares REPLICA's
`evaluate()` output against that stored anchor with tolerances appropriate to
the backend. Mark tests that read downloaded inputs with `needs_data`, tests
that load model weights with `needs_model`, and R-backed fits with `needs_r`.
Use `artifact` for indexed study evidence. The study's README identifies the
reference and the scope of the comparison.


## Where you edit, at a glance

| step | file |
|---|---|
| dataset loader | `dataset/<paradigm>/<name>.py` |
| dataset metadata | `dataset/metadata/yaml/<paradigm>/<name>.yaml` |
| dataset registration | `dataset/catalog/registry.py` (`_BASE_ENTRIES`) |
| dataset download | `dataset/catalog/sources.py` (`DATASETS`) |
| dataset counts | `dataset/catalog/dataset_counts.json` (`replica datasets --facts --measure --write-snapshot --only-missing`) |
| feature/model | its concept package + public export in `feature/__init__.py` |
| study recipe | `studies/<name>/__init__.py` |
| study README | `studies/<name>/README.md` (recipe sections) and `studies/<name>/EVIDENCE.md` (claims, scope, stages, notes) |
| study claims | `studies/<name>/claims.yaml`, then `scripts/readmes/regen_study_readmes.py --study <name>` |
| study registration | none: a subpackage with a README is a study; its class name lowercases to the package name |
| reproduction anchor | `tests/studies/<name>/test_reproduction.py` |

## Conventions

- Study naming is `lastname{year}{slug}`, the slug disambiguating same-author,
  same-year papers.
- `bibtex` is the single source of citation truth; `paper` renders from it.
- A dataset loader must set `info.language` explicitly for non-English corpora.
- Put the feature under test in `predictors` and everything else in `controls`;
  `delta_ll` and `delta_r2` summarize its held-out contribution.
- A passing reproduction test shows that REPLICA still matches its stored anchor;
  the anchor's source (the paper or an independent implementation) determines what
  that match establishes.